# Phase 40 — Qwen LoRA full run

Fresh thin controller. **STOP_PACKAGE_AUTHORITY**: before running any installation or model acquisition step, verify the exact package releases named by the frozen request and explicitly change both authority booleans. This notebook starts one fresh full run at optimizer step zero and delegates training, evaluation, evidence, and graphs to repository commands.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import io
import json
import stat
import subprocess
import sys
import zipfile

TRANSFER_REPOSITORY_ROOT = Path("/content/drive/MyDrive/internship-phase40/repository")
REQUEST_PATH = TRANSFER_REPOSITORY_ROOT / "data/models/phase40/full-run-request.json"
SOURCE_ARCHIVE_RELATIVE = "data/models/phase40/source/phase40-source.zip"
SOURCE_INVENTORY_RELATIVE = "data/models/phase40/source/phase40-source-manifest.json"
SOURCE_EXTRACTION_ROOT = Path("/content/phase40-source-v1")
INPUT_ARCHIVE_PATH = Path("/content/drive/MyDrive/internship-phase40/phase40-train-validation.zip")
INPUT_EXTRACTION_ROOT = Path("/content/phase40-input-v1")
INPUT_MEMBERS = ("phase40-input-manifest.json", "train.jsonl", "val.jsonl")
MODEL_FAMILY = "qwen"
ADAPTATION_MODE = "lora"
RUN_KIND = "full"
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
BASE_MODEL_RELATIVE = "data/models/phase40/base/qwen3-4b-instruct-2507"
BASE_MODEL_PATH = TRANSFER_REPOSITORY_ROOT / BASE_MODEL_RELATIVE
BASE_MODEL_MANIFEST_RELATIVE = "data/models/phase40/base/qwen3-4b-instruct-2507.provenance.json"
BASE_MODEL_MANIFEST_PATH = TRANSFER_REPOSITORY_ROOT / BASE_MODEL_MANIFEST_RELATIVE
EXPECTED_RETURNED_ROOT = "data/models/phase40/full/qwen-lora"
DEPENDENCY_PINS = (
    "torch==2.12.0+cu132",
    "transformers==5.9.0",
    "peft==0.19.1",
    "huggingface-hub==1.16.1",
    "accelerate==1.13.0",
    "pydantic==2.13.4",
    "pydantic-settings==2.14.1",
    "scikit-learn==1.8.0",
    "matplotlib==3.11.1",
)
MODE_NO_DEPS_PINS = ()
PACKAGE_INSTALL_PINS = tuple(pin for pin in DEPENDENCY_PINS if pin not in MODE_NO_DEPS_PINS)
CLI = (sys.executable, "-m", "src.model_adaptation.phase40_operator")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
PACKAGE_AUTHORITY_GRANTED = False
MODEL_ACQUISITION_AUTHORIZED = False
if PACKAGE_AUTHORITY_GRANTED is not True or MODEL_ACQUISITION_AUTHORIZED is not True:
    raise RuntimeError("STOP_PACKAGE_AUTHORITY: inspect and approve the exact package pins and pinned model acquisition before continuing")


In [ ]:
SOURCE_BUNDLE_VERIFIED = False
def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

request_payload = json.loads(REQUEST_PATH.read_text(encoding="utf-8", errors="strict"))
if request_payload.get("schema_version") != "phase40-full-run-request-v1":
    raise RuntimeError("unsupported full-run request schema")
source_reference = request_payload.get("source_bundle")
if not isinstance(source_reference, dict):
    raise RuntimeError("run request has no source_bundle authority")
if source_reference.get("repository_relative_archive_path") != SOURCE_ARCHIVE_RELATIVE:
    raise RuntimeError("source archive path differs from the canonical request path")
if source_reference.get("repository_relative_inventory_path") != SOURCE_INVENTORY_RELATIVE:
    raise RuntimeError("source inventory path differs from the canonical request path")
source_archive_path = TRANSFER_REPOSITORY_ROOT / SOURCE_ARCHIVE_RELATIVE
source_inventory_path = TRANSFER_REPOSITORY_ROOT / SOURCE_INVENTORY_RELATIVE
if TRANSFER_REPOSITORY_ROOT.resolve() not in source_archive_path.resolve().parents:
    raise RuntimeError("source archive escapes the transfer repository")
if TRANSFER_REPOSITORY_ROOT.resolve() not in source_inventory_path.resolve().parents:
    raise RuntimeError("source inventory escapes the transfer repository")
source_archive_bytes = source_archive_path.read_bytes()
source_inventory_bytes = source_inventory_path.read_bytes()
if sha256_bytes(source_archive_bytes) != source_reference.get("archive_sha256"):
    raise RuntimeError("source archive SHA-256 mismatch")
if sha256_bytes(source_inventory_bytes) != source_reference.get("inventory_sha256"):
    raise RuntimeError("source inventory SHA-256 mismatch")
source_inventory_payload = json.loads(source_inventory_bytes.decode("utf-8", errors="strict"))
if source_inventory_payload["archive_sha256"] != source_reference["archive_sha256"]:
    raise RuntimeError("source inventory names a different archive")
source_entries = source_reference.get("files")
if source_inventory_payload.get("files") != source_entries or not isinstance(source_entries, list):
    raise RuntimeError("source inventory differs from the frozen request")
source_names = [entry.get("path") for entry in source_entries]
if not source_names or source_names != sorted(source_names) or len(source_names) != len(set(source_names)):
    raise RuntimeError("source inventory must be non-empty, sorted, and unique")
verified_source_payloads = {}
with zipfile.ZipFile(io.BytesIO(source_archive_bytes), mode="r") as source_archive:
    source_infos = source_archive.infolist()
    if [info.filename for info in source_infos] != source_names:
        raise RuntimeError("source archive members differ from the frozen inventory")
    for info, entry in zip(source_infos, source_entries, strict=True):
        relative = PurePosixPath(info.filename)
        unix_kind = (info.external_attr >> 16) & 61440
        if relative.is_absolute() or ".." in relative.parts or info.is_dir() or unix_kind == stat.S_IFLNK:
            raise RuntimeError("unsafe source archive member")
        if info.compress_type != zipfile.ZIP_STORED:
            raise RuntimeError("source archive member is not deterministic stored data")
        member_payload = source_archive.read(info.filename)
        if len(member_payload) != entry.get("bytes") or sha256_bytes(member_payload) != entry.get("sha256"):
            raise RuntimeError("source archive member identity mismatch")
        verified_source_payloads[info.filename] = member_payload
SOURCE_BUNDLE_VERIFIED = True


In [ ]:
if SOURCE_BUNDLE_VERIFIED is not True:
    raise RuntimeError("source bundle must verify before extraction")
if SOURCE_EXTRACTION_ROOT.exists():
    raise RuntimeError("fresh source extraction root already exists; inspect it instead of overwriting")
SOURCE_EXTRACTION_ROOT.mkdir(parents=True, exist_ok=False)
for source_name in source_names:
    source_target = SOURCE_EXTRACTION_ROOT / PurePosixPath(source_name)
    if SOURCE_EXTRACTION_ROOT.resolve() not in source_target.parent.resolve().parents and source_target.parent.resolve() != SOURCE_EXTRACTION_ROOT.resolve():
        raise RuntimeError("source target escapes the verified extraction root")
    source_target.parent.mkdir(parents=True, exist_ok=True)
    source_target.write_bytes(verified_source_payloads[source_name])
sys.path.insert(0, str(SOURCE_EXTRACTION_ROOT))


In [ ]:
if PACKAGE_AUTHORITY_GRANTED is not True:
    raise RuntimeError("STOP_PACKAGE_AUTHORITY: package authority is required")
install_command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
install_command.extend(PACKAGE_INSTALL_PINS)
subprocess.run(install_command, check=True)
if MODE_NO_DEPS_PINS:
    mode_install_command = [sys.executable, "-m", "pip", "install", "--no-deps"]
    mode_install_command.extend(MODE_NO_DEPS_PINS)
    subprocess.run(mode_install_command, check=True)
import importlib.metadata as package_metadata
for dependency_pin in DEPENDENCY_PINS:
    dependency_name, dependency_version = dependency_pin.split("==", 1)
    if package_metadata.version(dependency_name) != dependency_version:
        raise RuntimeError("installed dependency differs from the exact pin")


In [ ]:
from src.model_adaptation.phase40_handoff import RunRequest, verify_phase40_run_request
typed_request = RunRequest.model_validate(request_payload)
verify_phase40_run_request(typed_request, repo_root=TRANSFER_REPOSITORY_ROOT, verify_input=False)
if typed_request.source_bundle.archive_sha256 != source_reference["archive_sha256"]:
    raise RuntimeError("typed source_bundle identity differs from bootstrap verification")
matching_runs = [item for item in typed_request.runs if item.model_family.value == MODEL_FAMILY and item.adaptation_mode.value == ADAPTATION_MODE]
if len(matching_runs) != 1:
    raise RuntimeError("run request does not contain exactly one positive experiment identity")
run_identity = matching_runs[0]
if run_identity.run_kind != RUN_KIND or run_identity.step_origin != 0 or run_identity.probe_parent is not None:
    raise RuntimeError("full run must be fresh at step zero with no parent")
if run_identity.returned_root != EXPECTED_RETURNED_ROOT:
    raise RuntimeError("run request returned root is not canonical")
RUN_ID = run_identity.run_id
CONTROL_TEMPLATE = typed_request.control_template_by_run[RUN_ID]
CONTROL_VALUES = CONTROL_TEMPLATE.controls_without_accelerator
if CONTROL_VALUES.get("model_id") != MODEL_ID:
    raise RuntimeError("run-request model ID differs from the notebook pin")
if CONTROL_VALUES.get("model_revision") != MODEL_REVISION:
    raise RuntimeError("run-request model revision differs from the notebook pin")
if CONTROL_VALUES.get("experiment_identity", {}).get("adaptation_mode") != ADAPTATION_MODE:
    raise RuntimeError("run-request adaptation mode differs from the notebook")
input_authority = typed_request.input_bundle
RUN_OUTPUT_ROOT = Path("/content/drive/MyDrive/internship-phase40/work") / RUN_ID
RETURNED_BUNDLE_ROOT = TRANSFER_REPOSITORY_ROOT / run_identity.returned_root
def run_cli(arguments):
    subprocess.run(list(CLI) + list(arguments), cwd=SOURCE_EXTRACTION_ROOT, check=True)


In [ ]:
MODEL_SNAPSHOT_VERIFIED = False
if MODEL_ACQUISITION_AUTHORIZED is not True:
    raise RuntimeError("pinned model acquisition lacks explicit operator authority")
run_cli(("phase40-acquire-model", "--request-path", str(REQUEST_PATH), "--repo-root", str(TRANSFER_REPOSITORY_ROOT), "--run-id", RUN_ID, "--authorize-model-acquisition"))
if not BASE_MODEL_PATH.is_dir() or not BASE_MODEL_MANIFEST_PATH.is_file():
    raise RuntimeError("sealed base-model snapshot or provenance manifest is missing")
MODEL_SNAPSHOT_VERIFIED = True


In [ ]:
INPUT_BUNDLE_VERIFIED = False
if input_authority.drive_path != str(INPUT_ARCHIVE_PATH) or input_authority.extraction_root != str(INPUT_EXTRACTION_ROOT):
    raise RuntimeError("input Drive or extraction path differs from the fixed request")
if tuple(input_authority.members) != INPUT_MEMBERS:
    raise RuntimeError("input bundle does not contain the exact three members")
for input_hash in (input_authority.archive_sha256, input_authority.manifest_sha256, input_authority.phase39_data_contract_sha256):
    if len(input_hash) != 64 or any(character not in "0123456789abcdef" for character in input_hash):
        raise RuntimeError("input archive_sha256 or manifest_sha256 field is invalid")
for data_member in input_authority.data_members:
    if len(data_member.sha256) != 64 or len(data_member.ordered_row_ids_sha256) != 64:
        raise RuntimeError("input member hash or ordered_row_ids_sha256 is invalid")
if input_authority.snapshot_row_id_version != "phase40-snapshot-row-id-v1":
    raise RuntimeError("input snapshot_row_id_version is not canonical")
control_root = Path("/content/phase40-control")
control_root.mkdir(parents=True, exist_ok=True)
input_reference_path = control_root / f"{RUN_ID}-input-reference.json"
input_reference_path.write_text(input_authority.model_dump_json(indent=2) + "\n", encoding="utf-8")
run_cli(("phase40-verify-input-bundle", "--archive-path", str(INPUT_ARCHIVE_PATH), "--reference-path", str(input_reference_path), "--repo-root", str(TRANSFER_REPOSITORY_ROOT), "--extraction-root", str(INPUT_EXTRACTION_ROOT)))
INPUT_BUNDLE_VERIFIED = True


In [ ]:
run_cli(("phase40-doctor", "--model-family", MODEL_FAMILY, "--adaptation-mode", ADAPTATION_MODE, "--run-kind", RUN_KIND, "--model-revision", MODEL_REVISION, "--run-request-path", str(REQUEST_PATH), "--input-root", str(INPUT_EXTRACTION_ROOT), "--base-model-path", str(BASE_MODEL_PATH), "--base-model-manifest-path", str(BASE_MODEL_MANIFEST_PATH)))


In [ ]:
EXACT_RESUME_CHECKPOINT = None
EXACT_RESUME_VERIFIED = EXACT_RESUME_CHECKPOINT is None
if EXACT_RESUME_CHECKPOINT is None and RUN_OUTPUT_ROOT.exists():
    raise RuntimeError("fresh step-zero work root already exists; use only an exact verified resume or a fresh run identity")
if EXACT_RESUME_CHECKPOINT is not None:
    run_cli(("phase40-verify-resume", "--request-path", str(REQUEST_PATH), "--run-id", RUN_ID, "--checkpoint", str(EXACT_RESUME_CHECKPOINT), "--input-root", str(INPUT_EXTRACTION_ROOT), "--base-model-path", str(BASE_MODEL_PATH), "--base-model-manifest-path", str(BASE_MODEL_MANIFEST_PATH)))
    EXACT_RESUME_VERIFIED = True


In [ ]:
if MODEL_ACQUISITION_AUTHORIZED is not True or MODEL_SNAPSHOT_VERIFIED is not True or INPUT_BUNDLE_VERIFIED is not True or EXACT_RESUME_VERIFIED is not True:
    raise RuntimeError("full run authority is incomplete")
run_arguments = ["phase40-train-qwen", "--adaptation-mode", "lora", "--run-kind", "full", "--request-path", str(REQUEST_PATH), "--repo-root", str(TRANSFER_REPOSITORY_ROOT), "--input-archive", str(INPUT_ARCHIVE_PATH), "--extraction-root", str(INPUT_EXTRACTION_ROOT), "--run-id", RUN_ID, "--output-root", str(RUN_OUTPUT_ROOT), "--base-model-path", str(BASE_MODEL_PATH), "--base-model-manifest-path", str(BASE_MODEL_MANIFEST_PATH)]
if EXACT_RESUME_CHECKPOINT is not None:
    if EXACT_RESUME_VERIFIED is not True:
        raise RuntimeError("resume compatibility was not verified")
    run_arguments.extend(("--resume-from-checkpoint", str(EXACT_RESUME_CHECKPOINT)))
run_cli(run_arguments)


In [ ]:
run_cli(("phase40-verify-run-evidence", "--run-root", str(RETURNED_BUNDLE_ROOT)))


In [ ]:
run_cli(("phase40-render-graphs", "--run-root", str(RETURNED_BUNDLE_ROOT)))
run_cli(("phase40-verify-run-evidence", "--run-root", str(RETURNED_BUNDLE_ROOT)))


In [ ]:
run_cli(("phase40-verify-run-evidence", "--run-root", str(RETURNED_BUNDLE_ROOT)))
print(f"Verified request-bound bundle ready for unchanged return: {RETURNED_BUNDLE_ROOT}")
